# A/B Testing: Hypothesis Testing Ships to Production

The lesson's promised destination: **A/B testing**, the methodology behind every "which button converts better?" decision on the web. Customers are randomly split between variant A and variant B (traditionally with the *smaller* share going to the untested newcomer), an outcome metric is chosen, and — as the final step — the statistical tests of this lesson render the verdict. Two full experiments: comparing **purchase amounts** (80 vs 20 customers, `$50` vs `$55`) with the Welch two-sample t-test ($t = -1.414$, $\nu = 23.38$, $p = 0.085$ — not enough evidence), and comparing **conversion rates** (20/80 vs 8/20 purchases) with a new tool assembled live from Bernoulli parts — the pooled **two-proportion z-test** ($z = -1.336$, $p = 0.091$ — also not enough). It is based on the twelfth and final video of Lesson 8 (Hypothesis Testing), which closes the lesson, the week, and the specialization's lecture arc.

## 1. Introduction

Every tool of the last eleven notebooks reports for duty here: hypotheses and tails, $\alpha$, p-values, the two-sample t-machinery, and the Bernoulli/CLT bridge from the confidence-intervals lesson. A/B testing wraps them in an experimental protocol — and the two worked examples deliver a final, very practical lesson about sample sizes and honest conclusions.

After completing this notebook you will be able to:

- Describe the A/B methodology end to end — variations, random split, metric choice, measurement, statistical decision — and where the t-test fits (and where it doesn't).
- Run the purchase-amount experiment as a Welch two-sample t-test and reproduce $t = -1.414$, $p = 0.085$.
- Derive the pooled two-proportion z-statistic from binomial first principles, and reproduce $z = -1.336$, $p = 0.091$.
- Report both non-rejections honestly — and diagnose the 80/20 split's role in them.
- Probe the practice: how splits affect power, and why peeking at running experiments corrupts $\alpha$.

## 2. Theory

### The methodology

A/B testing is **broader than a t-test** — it is a methodology for comparing two variations, of which statistics is the last step:

1. **Propose the variations** — e.g. two placements of the *Buy Now* button, strategy A (current) and B (new).
2. **Randomly split** incoming customers between them. The common rule: send a **smaller proportion to the new design**, since you don't yet know how good (or bad) it is — here 80 customers to A, 20 to B.
3. **Measure outcomes and choose the metric** — mean behavior? proportions? dispersion? This choice decides which statistical test applies.
4. **Use a statistical test** on the gathered data to make the decision. For Gaussian means, that's the two-sample t-test; for other metrics, other tests (as the second experiment shows).

### Experiment 1: purchase amounts (a t-test job)

| group | $n$ | mean purchase | sample sd |
|---|---|---|---|
| A | 80 | `$50` | `$10` |
| B | 20 | `$55` | `$15` |

Hypotheses (does B raise purchases?): $H_0: \mu_A = \mu_B$ vs $H_1: \mu_B > \mu_A$ — in difference form, $\mu_A - \mu_B < 0$: a **left-tailed** test at $\alpha = 0.05$. Assuming Gaussian purchase amounts (or enough samples), the Welch two-sample t-test applies:

$$
t_{obs} = \frac{50 - 55}{\sqrt{\frac{10^2}{80} + \frac{15^2}{20}}} = \frac{-5}{3.536} = -1.414,
\qquad \nu = 23.38
$$

$$
p = P(T < -1.414) = 0.085 > 0.05 \quad\Rightarrow\quad \text{do not reject } H_0
$$

The `$5` gap in favor of B is *suggestive but not established* — B's 20 noisy customers ($s = 15$) leave too much doubt.

### Experiment 2: conversion rates (a new statistic, assembled live)

Now test a whole new site design on the **conversion rate** — the proportion of visitors who purchase. Data: group A, 80 visitors, 20 purchases ($\hat{p}_A = 0.25$); group B, 20 visitors, 8 purchases ($\hat{p}_B = 0.40$). Hypotheses: $H_0: p_A = p_B$ vs $H_1: p_B > p_A$ (left-tailed in $p_A - p_B$), $\alpha = 0.05$.

**Building the statistic.** $X \sim \text{Bin}(n_A, p_A)$ and $Y \sim \text{Bin}(n_B, p_B)$ are sums of Bernoullis, so by the law of large numbers $X/n_A \approx p_A$, and by the CLT

$$
\frac{X}{n_A} \approx N\!\left(p_A, \frac{p_A(1-p_A)}{n_A}\right), \qquad
\frac{Y}{n_B} \approx N\!\left(p_B, \frac{p_B(1-p_B)}{n_B}\right)
$$

Their difference is then approximately Gaussian with mean $p_A - p_B$ and the variances summed. Standardize — and now use $H_0$, under which $p_A = p_B = p$: the variance terms share the common factor $p(1-p)$, and $\frac{1}{n_A} + \frac{1}{n_B} = \frac{n_A + n_B}{n_A n_B}$, giving

$$
Z = \frac{\frac{X}{n_A} - \frac{Y}{n_B}}{\sqrt{p(1-p)\cdot\frac{n_A + n_B}{n_A n_B}}} \;\approx\; N(0, 1)
$$

One unknown remains: $p$ itself. Since under $H_0$ both groups share it, the best estimate **pools all observations**:

$$
\hat{p} = \frac{X + Y}{n_A + n_B}
$$

— treating the combined $n_A + n_B$ visitors as one Bernoulli sample. Replacing $p(1-p)$ with $\hat{p}(1-\hat{p})$ completes the **two-proportion z-test** statistic.

**The verdict.** $\hat{p} = 28/100 = 0.28$, and

$$
z_{obs} = \frac{0.25 - 0.40}{\sqrt{0.28 \times 0.72 \times \frac{100}{1600}}} = -1.336, \qquad
p = P(Z < -1.336) = 0.091
$$

Since $0.091 > 0.05$: **do not reject $H_0$** — the evidence is insufficient to establish that design B converts better. (The video misspeaks "reject" here; its own comparison — p-value bigger than the significance level — settles the verdict.) Again B *looks* better (40% vs 25%), and again 20 visitors are too few to prove it.

## 3. Worked Examples

### Example 1: the t-test arithmetic

$SE = \sqrt{100/80 + 225/20} = \sqrt{1.25 + 11.25} = \sqrt{12.5} = 3.536$; $t = -5/3.536 = -1.414$; Welch $\nu = 12.5^2 / (1.25^2/79 + 11.25^2/19) = 23.38$. Note where the SE comes from: almost all of it ($11.25$ of $12.5$) is contributed by B's small, noisy group.

### Example 2: the pooled z arithmetic

$\hat{p} = (20 + 8)/(80 + 20) = 0.28$; $SE = \sqrt{0.28 \times 0.72 \times 100/1600} = \sqrt{0.0126} = 0.1122$; $z = (0.25 - 0.40)/0.1122 = -1.336$; left tail of the standard normal: $0.091$.

### Example 3: reading the pair of experiments like a data scientist

Both metrics favor B (`$55` vs `$50`; 40% vs 25%), both tests decline to certify it, and both for the same structural reason: the cautious 80/20 split starves the B group. That caution is *deliberate* — step 2's rule protects revenue from a possibly-bad variant — but it trades statistical power for safety. The honest report: "B trends better on both metrics; evidence insufficient at $\alpha = 0.05$; recommend extending the experiment" — and the exercises quantify exactly how much more data (or how different a split) would settle it.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project) and both test engines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from math import gamma, sqrt, pi

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(12)

Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))

def t_pdf(t, df):
    coeff = gamma((df + 1) / 2) / (sqrt(df * pi) * gamma(df / 2))
    return coeff * (1 + np.asarray(t, dtype=float) ** 2 / df) ** (-(df + 1) / 2)

def t_tail(t_value, df):
    grid = np.linspace(0, abs(t_value), 20_000)
    tail = 0.5 - np.trapezoid(t_pdf(grid, df), grid)
    return tail if t_value >= 0 else 1 - tail

def welch_from_stats(m1, s1, n1, m2, s2, n2, tail="left"):
    """Welch two-sample t-test from summary statistics."""
    v1, v2 = s1**2 / n1, s2**2 / n2
    t_obs = (m1 - m2) / np.sqrt(v1 + v2)
    df = (v1 + v2) ** 2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))
    p = {"right": t_tail(t_obs, df), "left": 1 - t_tail(t_obs, df),
         "two": 2 * t_tail(abs(t_obs), df)}[tail]
    return t_obs, df, p

def two_proportion_test(x, n_a, y, n_b, tail="left"):
    """Pooled two-proportion z-test for H0: p_A = p_B."""
    p_pool = (x + y) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (n_a + n_b) / (n_a * n_b))
    z_obs = (x / n_a - y / n_b) / se
    p = {"right": 1 - Phi(z_obs), "left": Phi(z_obs),
         "two": 2 * (1 - Phi(abs(z_obs)))}[tail]
    return z_obs, p

### Experiment 1: the Buy Now button

In [ ]:
t_obs, df_w, p1 = welch_from_stats(50, 10, 80, 55, 15, 20, tail="left")
print("H0: mu_A = mu_B   H1: mu_B > mu_A (left-tailed in mu_A - mu_B)")
print(f"t_obs = {t_obs:.3f}, Welch df = {df_w:.2f}, p = {p1:.3f}")
print(f"verdict at alpha = 0.05: {'REJECT' if p1 < 0.05 else 'do NOT reject'} --")
print("the $5 advantage of B is not statistically established.")

### Experiment 2: the site redesign's conversion rate

In [ ]:
x, n_a = 20, 80          # design A: 20 of 80 convert
y, n_b = 8, 20           # design B:  8 of 20 convert

p_pool = (x + y) / (n_a + n_b)
print(f"p-hat_A = {x / n_a:.2f}, p-hat_B = {y / n_b:.2f}, pooled p-hat = {p_pool:.2f}")

z_obs, p2 = two_proportion_test(x, n_a, y, n_b, tail="left")
print(f"z_obs = {z_obs:.3f}, p = P(Z < {z_obs:.3f}) = {p2:.3f}")
print(f"verdict at alpha = 0.05: {'REJECT' if p2 < 0.05 else 'do NOT reject'} --")
print("B's 40% vs 25% conversion is suggestive, but 20 visitors prove nothing.")

### Is the pooled statistic really standard normal? A simulation check

Under $H_0$ (one shared conversion rate), simulate the whole experiment many times and compare the statistic's histogram — and its realized Type I rate — with $N(0,1)$.

In [ ]:
trials = 100_000
p_true = 0.28
x_sim = rng.binomial(80, p_true, trials)
y_sim = rng.binomial(20, p_true, trials)
p_pool_sim = (x_sim + y_sim) / 100
se_sim = np.sqrt(p_pool_sim * (1 - p_pool_sim) * 100 / 1600)
z_sim = (x_sim / 80 - y_sim / 20) / se_sim

z_axis = np.linspace(-4, 4, 400)
plt.figure(figsize=(9.5, 4.4))
plt.hist(z_sim, bins=120, range=(-4, 4), density=True, color="tab:green",
         alpha=0.55, label="simulated pooled z under $H_0$")
plt.plot(z_axis, np.exp(-0.5 * z_axis**2) / np.sqrt(2 * np.pi), color="black",
         linewidth=2, label="N(0, 1)")
plt.xlabel("z")
plt.ylabel("density")
plt.title("The Bernoulli-CLT-pooling construction delivers a standard normal")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

z05 = float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(0.05) - 1))
print(f"realized Type I rate of 'reject when z < {z05:.3f}': "
      f"{(z_sim < z05).mean():.3f} (target 0.05; slight graininess is the")
print("binomial's discreteness at these sample sizes)")

## 5. Visualizations

### The experiment dashboard

What the data science team sees: both metrics lean B, both intervals overlap generously.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

means = [50, 55]
sems = [10 / np.sqrt(80), 15 / np.sqrt(20)]
axes[0].bar(["A (n = 80)", "B (n = 20)"], means, yerr=[1.96 * s for s in sems],
            color=["tab:blue", "tab:orange"], alpha=0.8, capsize=8, width=0.55)
axes[0].set_ylabel("mean purchase ($)")
axes[0].set_ylim(40, 65)
axes[0].set_title(f"purchase amounts: p = {p1:.3f}", fontsize=10)

rates = [x / n_a, y / n_b]
rate_ses = [np.sqrt(r * (1 - r) / n) for r, n in zip(rates, [n_a, n_b])]
axes[1].bar(["A (20/80)", "B (8/20)"], rates, yerr=[1.96 * s for s in rate_ses],
            color=["tab:blue", "tab:orange"], alpha=0.8, capsize=8, width=0.55)
axes[1].set_ylabel("conversion rate")
axes[1].set_ylim(0, 0.7)
axes[1].set_title(f"conversion rates: p = {p2:.3f}", fontsize=10)

for ax in axes:
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("Both metrics favor B; neither difference clears alpha = 0.05 -- the B bars' error whiskers tell you why")
plt.tight_layout()
plt.show()

### The two verdicts on their two curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

t_axis = np.linspace(-4.5, 4.5, 600)
axes[0].plot(t_axis, t_pdf(t_axis, df_w), color="black", linewidth=1.6)
axes[0].fill_between(t_axis, t_pdf(t_axis, df_w), where=t_axis <= t_obs,
                     color="tab:red", alpha=0.55)
axes[0].axvline(t_obs, color="tab:red", linewidth=1.8)
axes[0].set_title(f"experiment 1: $t_{{23.38}}$, p = {p1:.3f} -> don't reject",
                  fontsize=10)
axes[0].set_xlabel("t")
axes[0].set_ylabel("density under $H_0$")

axes[1].plot(z_axis, np.exp(-0.5 * z_axis**2) / np.sqrt(2 * np.pi),
             color="black", linewidth=1.6)
axes[1].fill_between(z_axis, np.exp(-0.5 * z_axis**2) / np.sqrt(2 * np.pi),
                     where=z_axis <= z_obs, color="tab:red", alpha=0.55)
axes[1].axvline(z_obs, color="tab:red", linewidth=1.8)
axes[1].set_title(f"experiment 2: N(0,1), p = {p2:.3f} -> don't reject",
                  fontsize=10)
axes[1].set_xlabel("z")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.axvline(0, color="black", linestyle=":", linewidth=1)
plt.suptitle("Two experiments, two statistics, one honest conclusion: not enough evidence yet")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Rerun experiment 1 two-tailed ($H_1: \mu_A \ne \mu_B$). What is the p-value, and in what business situation would the two-tailed question be the right one to ask?

**Exercise 2.** Map the A/B methodology onto a concrete scenario — testing a new email subject line for a newsletter — naming all four steps, the metric type, and the statistical test the last step would use.

### Intermediate

**Exercise 3.** *Same rates, four times the traffic.* Repeat the conversion test with 320 visitors on A (80 conversions) and 80 on B (32) — identical rates. Compute the new $z$ and p-value. What single sentence summarizes what changed?

**Exercise 4.** *Honest reporting drill.* Write the one-sentence conclusion of experiment 2 three ways: (a) the video's misspoken version; (b) the statistically correct verdict; (c) the version a good data scientist would put in the deck (verdict + effect sizes + recommendation). Explain what (c) adds beyond (b).

### Advanced

**Exercise 5.** *The price of the safe split.* At a fixed total of 100 customers with true rates $p_A = 0.25$, $p_B = 0.40$, simulate the left-tailed test's power for B-shares of 10%, 20%, 30%, 50%, and 70%. Where does power peak, how costly was the video's 80/20, and why might a company accept that cost anyway?

**Exercise 6.** *Peeking corrupts $\alpha$.* Simulate a null A/B experiment ($p_A = p_B = 0.28$, 50/50 split) that a manager checks after every 40 additional visitors up to 400, stopping the moment $p < 0.05$. Estimate the true false-positive rate of this "peek and stop" habit and compare it to the promised 5%. What does this imply about running A/B tests to a fixed horizon?

## 7. Solutions

In [ ]:
# Exercise 1: the two-tailed version
t1, df1, p1_two = welch_from_stats(50, 10, 80, 55, 15, 20, tail="two")
print(f"two-tailed p = {p1_two:.3f} (double the one-tailed 0.085)")
print("\nTwo-tailed is right when EITHER direction changes the decision -- e.g.")
print("if strategy B is costlier to run, a decrease in purchases would also")
print("matter, so 'did B change purchases at all?' is the honest question.")

In [ ]:
# Exercise 2: the methodology mapped
print("1. variations: current subject line (A) vs new subject line (B).")
print("2. random split: each outgoing email randomly assigned, smaller share to B.")
print("3. metric: open rate -- a PROPORTION (opened / delivered).")
print("4. statistical decision: the pooled two-proportion z-test of this notebook")
print("   (means machinery would be wrong: the outcome is binary, not Gaussian).")

In [ ]:
# Exercise 3: quadrupled traffic, same rates
z3, p3 = two_proportion_test(80, 320, 32, 80, tail="left")
print(f"z = {z3:.3f}, p = {p3:.4f} -> REJECT at alpha = 0.05")
print("\nOne sentence: the effect didn't change, the evidence did -- the same 15-")
print("point conversion gap that 100 visitors couldn't certify, 400 visitors can.")

In [ ]:
# Exercise 4: three conclusions
print("(a) misspoken: 'p = 0.091 > 0.05, so we reject H0 and conclude B converts")
print("    better.'  (verdict contradicts its own comparison)")
print("(b) correct: 'p = 0.091 > 0.05: we fail to reject H0 -- the evidence does")
print("    not establish that B converts better.'")
print("(c) deck-ready: 'B converted 40% vs A's 25%, a promising 15-point lift,")
print("    but with only 20 visitors on B the difference is not yet significant")
print("    (p = 0.09); recommend extending the test toward ~400 total visitors.'")
print("\n(c) adds the effect size and the ACTION -- a non-rejection is a request")
print("for more data, not a conclusion that B is worthless.")

In [ ]:
# Exercise 5: power vs split
trials = 40_000
total = 100
z05 = float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(0.05) - 1))
print(f"{'B share':>8s} {'n_B':>5s} {'power':>7s}")
for share in (0.10, 0.20, 0.30, 0.50, 0.70):
    nb_ = int(total * share)
    na_ = total - nb_
    xs = rng.binomial(na_, 0.25, trials)
    ys = rng.binomial(nb_, 0.40, trials)
    pp = (xs + ys) / total
    se = np.sqrt(pp * (1 - pp) * total / (na_ * nb_))
    zz = (xs / na_ - ys / nb_) / se
    print(f"{share:>8.0%} {nb_:>5d} {(zz < z05).mean():>7.1%}")
print("\nPower peaks near 50/50 (balanced information) and the video's 80/20")
print("gives up roughly a third of it. The cost can still be worth paying: the")
print("split caps how many customers a genuinely BAD variant can hurt.")

In [ ]:
# Exercise 6: peek-and-stop inflation
trials = 4_000
false_positives = 0
for _ in range(trials):
    rejected = False
    xa = ya = na_ = nb_ = 0
    for _ in range(10):                       # 10 looks, 40 visitors each
        xa += rng.binomial(20, 0.28); na_ += 20
        ya += rng.binomial(20, 0.28); nb_ += 20
        pp = (xa + ya) / (na_ + nb_)
        if pp in (0, 1):
            continue
        se = np.sqrt(pp * (1 - pp) * (na_ + nb_) / (na_ * nb_))
        zz = (xa / na_ - ya / nb_) / se
        if Phi(zz) < 0.05:                    # left-tailed peek
            rejected = True
            break
    false_positives += rejected
print(f"promised Type I rate: 5.0%")
print(f"realized rate with 10 peeks: {false_positives / trials:.1%}")
print("\nEvery peek is another lottery ticket against H0; ten of them roughly")
print("triples the false-positive rate. Fix the horizon before starting (or use")
print("sequential-testing corrections) -- the design discipline of notebook 05,")
print("one last time.")

## 8. Conclusion

- **A/B testing** is a methodology, not a single test: propose variations → randomly split (smaller share to the new design) → choose the metric → measure → and only then apply the matching statistical test.
- **Purchase amounts** (Gaussian means) → Welch two-sample t-test: $t = -1.414$, $\nu = 23.38$, $p = 0.085$ — B's `$5` edge not established.
- **Conversion rates** (proportions) → a statistic built from Bernoulli + CLT + $H_0$-pooling: $Z = (\hat{p}_A - \hat{p}_B)\big/\sqrt{\hat{p}(1-\hat{p})\tfrac{n_A + n_B}{n_A n_B}}$ with $\hat{p}$ pooled over all visitors: $z = -1.336$, $p = 0.091$ — B's 15-point lift not established either (the video's "reject" here is a slip; its own $p > \alpha$ comparison settles it).
- The practical morals, quantified in the exercises: the same rates with 4× traffic *do* reject ($p = 0.004$); the safe 80/20 split costs real power versus 50/50; and peeking at a running test triples the false-positive rate — fix the horizon in advance.
- And with that, the arc that began with ham and spam closes: hypotheses, errors, $\alpha$, p-values, critical values, power, t-tests in every flavor, and their deployment in production experiments.

This concludes Lesson 8 — and the specialization's lecture material. What remains in the course: the final exploratory data analysis lab, the summative quiz, and a graded programming assignment applying exactly this notebook's A/B machinery. Good luck!